In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/bernama_baseline_raw.csv')
print(df.info())
print(df.describe(include='all'))
print(df.isnull().sum())
print(df.duplicated().sum())
print(df['section'].value_counts())
print(df['publication_date'].sample(10))
print(df['headline'].str.len().describe())        # check headline lengths
print(df['article_summary'].str.len().describe()) # check summary lengths

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101212 entries, 0 to 101211
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   headline          101212 non-null  object
 1   publication_date  101212 non-null  object
 2   section           101212 non-null  object
 3   article_summary   101212 non-null  object
 4   url               101212 non-null  object
dtypes: object(5)
memory usage: 3.9+ MB
None
                          headline  publication_date section  \
count                       101212            101212  101212   
unique                       97597             96967       1   
top     Gold futures open untraded  2021-09-28 23:17    News   
freq                           173                 5  101212   

                                          article_summary  \
count                                              101212   
unique                                             100965   
top     BERN

In [8]:
# remove duplicates

df.drop_duplicates(subset='url', inplace=True)
df.drop_duplicates(subset='headline', keep='first', inplace=True)

In [9]:
# strip whitespace from all string columns

str_cols = ['headline', 'section', 'article_summary', 'url']
for col in str_cols:
    df[col] = df[col].str.strip()

In [10]:
# standardize publication_date

df['publication_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
# Check how many failed to parse
print(df['publication_date'].isna().sum())
df.dropna(subset=['publication_date'], inplace=True)
df['publication_date'] = df['publication_date'].dt.strftime('%Y-%m-%d %H:%M')

0


In [11]:
# validate url format

valid_url_mask = df['url'].str.startswith('https://www.bernama.com')
print(f"Invalid URLs: {(~valid_url_mask).sum()}")
df = df[valid_url_mask]

Invalid URLs: 0


In [12]:
# handle short summary

df = df[df['article_summary'].str.len() >= 40]

In [13]:
# standardize text casing for section

df['section'] = df['section'].str.title()

In [14]:
# add derived columns

df['pub_date_only'] = pd.to_datetime(df['publication_date']).dt.date
df['pub_year'] = pd.to_datetime(df['publication_date']).dt.year
df['pub_month'] = pd.to_datetime(df['publication_date']).dt.month
df['headline_word_count'] = df['headline'].str.split().str.len()

In [15]:
# Measure Baseline Performance

import time
import psutil
import os

process = psutil.Process(os.getpid())

# --- START BENCHMARK ---
start_time = time.time()
mem_before = process.memory_info().rss / (1024 * 1024)  # MB
cpu_before = psutil.cpu_percent(interval=1)

# ... run all your cleaning steps here ...

end_time = time.time()
mem_after = process.memory_info().rss / (1024 * 1024)
cpu_after = psutil.cpu_percent(interval=1)
# --- END BENCHMARK ---

total_time = end_time - start_time
records_cleaned = len(df)
throughput = records_cleaned / total_time

print(f"Total Time: {total_time:.2f}s")
print(f"Records Processed: {records_cleaned}")
print(f"Throughput: {throughput:.0f} records/sec")
print(f"Memory Used: {mem_after - mem_before:.1f} MB")
print(f"CPU Usage: {cpu_after:.1f}%")

Total Time: 1.00s
Records Processed: 97597
Throughput: 97504 records/sec
Memory Used: 0.0 MB
CPU Usage: 2.5%


In [16]:
# Save as CSV (primary deliverable)
df.to_csv('bernama_cleaned.csv', index=False)

# Optionally also save as JSON for variety
df.to_json('bernama_cleaned.json', orient='records', lines=True)

print(f"Final cleaned records: {len(df)}")

Final cleaned records: 97597


In [17]:
import pandas as pd

perf_before = pd.DataFrame([{
    'technique': 'Baseline (Single-threaded)',
    'total_time_sec': total_time,
    'records_processed': records_cleaned,
    'throughput_rec_per_sec': throughput,
    'memory_mb': mem_after - mem_before,
    'cpu_percent': cpu_after
}])
perf_before.to_csv('performance_before.csv', index=False)